In [8]:
import pandas as pd
import numpy as np
import sqlite3

In [2]:
url = 'https://raw.githubusercontent.com/melthawt/Loan-Prediction-Dataset/master/train.csv'

In [5]:

# Let's use an alternative source for the Loan Prediction dataset
url = 'https://raw.githubusercontent.com/dphi-official/Datasets/master/Loan_Data/loan_train.csv'

# Load the dataset
df = pd.read_csv(url)
print(f"✅ Dataset loaded: {df.shape}")
print(df.head())

# Save to local file
df.to_csv('loan_data_raw.csv', index=False)

✅ Dataset loaded: (491, 14)
   Unnamed: 0   Loan_ID  Gender Married Dependents     Education  \
0           0  LP002305  Female      No          0      Graduate   
1           1  LP001715    Male     Yes         3+  Not Graduate   
2           2  LP002086  Female     Yes          0      Graduate   
3           3  LP001136    Male     Yes          0  Not Graduate   
4           4  LP002529    Male     Yes          2      Graduate   

  Self_Employed  ApplicantIncome  CoapplicantIncome  LoanAmount  \
0            No             4547                0.0       115.0   
1           Yes             5703                0.0       130.0   
2            No             4333             2451.0       110.0   
3           Yes             4695                0.0        96.0   
4            No             6700             1750.0       230.0   

   Loan_Amount_Term  Credit_History Property_Area  Loan_Status  
0             360.0             1.0     Semiurban            1  
1             360.0           

In [10]:
# Load to SQLite (your strength!)
conn = sqlite3.connect('credit_risk.db')
df.to_sql('loans', conn, if_exists='replace', index=False)
conn.close()
print(" Database created: credit_risk.db")


 Database created: credit_risk.db


In [12]:
conn = sqlite3.connect('credit_risk.db')

# Feature Engineering
features_sql = """
WITH customer_features AS (
  SELECT 
    Loan_ID,
    ApplicantIncome,
    CoapplicantIncome,
    LoanAmount,
    Credit_History,
    Loan_Status,  -- Added Loan_Status column to the CTE
    Property_Area,
    
    -- Senior features
    ApplicantIncome + CoapplicantIncome as Total_Income,
    NTILE(5) OVER (ORDER BY ApplicantIncome DESC) as Income_Bucket,
    CASE 
      WHEN Credit_History = 1.0 THEN 'Good'
      WHEN Credit_History IS NULL THEN 'Unknown'
      ELSE 'Poor'
    END as Credit_Category,
    
    AVG(LoanAmount) OVER (PARTITION BY Property_Area) as Avg_Area_Loan
  FROM loans
)
SELECT 
  Loan_ID, Total_Income, Income_Bucket, Credit_Category, Avg_Area_Loan,
  Loan_Status,
  COUNT(*) OVER (PARTITION BY Credit_Category) as Category_Count
FROM customer_features;
"""

df_features = pd.read_sql_query(features_sql, conn)
df_features.to_csv('loan_features.csv', index=False)
print("Features created!")
print(df_features.head())
print(df_features['Loan_Status'].value_counts())

conn.close()

Features created!
    Loan_ID  Total_Income  Income_Bucket Credit_Category  Avg_Area_Loan  \
0  LP002101       63337.0              1            Good     143.397260   
1  LP001585       51763.0              1            Good     143.397260   
2  LP001640       43897.0              1            Good     141.902174   
3  LP002422       37719.0              1            Good     141.902174   
4  LP001448       23803.0              1            Good     150.593103   

   Loan_Status  Category_Count  
0            1             380  
1            1             380  
2            1             380  
3            1             380  
4            1             380  
Loan_Status
1    343
0    148
Name: count, dtype: int64
